In [2]:
R,C=map(int,input(" Enter value of R and C: ").split())
board=[]
for i in range(R):
    row=input("Enter row {}: ".format(i+1)).split()
    board.append(row)
    
starting_player=input("Enter starting player (MAX OR MIN): ").strip().upper()
# A is MAX player and B is MIN player

depth=int(input("Enter upto how much depth you want to search: "))

In [17]:
A_position=None
B_position=None

# We use a set as it is more efficient to check if a cell is in the set or not.
remaining_energy_cells=set() 

for i in range(R):
    for j in range(C):
        if board[i][j]=='A':
            A_position=(i,j)
        elif board[i][j]=='B':
            B_position=(i,j)
        elif board[i][j]=='E':
            remaining_energy_cells.add((i,j))

In [18]:
max_score=0
min_score=0
current_player=starting_player

initial_state=(A_position,B_position,remaining_energy_cells,current_player,max_score,min_score)


In [19]:
moves={"UP":(-1,0),"RIGHT":(0,1),"DOWN":(1,0),"LEFT":(0,-1)}

def get_new_position(position,move):
    new_row=position[0]+moves[move][0]
    new_col=position[1]+moves[move][1]
    if 0<=new_row<R and 0<=new_col<C:
        return (new_row,new_col)
    else:
        return None
    
def is_valid_move(new_pos,other_robot_pos):
    if new_pos is None:
        return False
    if new_pos==other_robot_pos:
        return False
    if board[new_pos[0]][new_pos[1]]=='#':
        return False
    return True

In [20]:
def successor(state,move):
    A_pos,B_pos,remaining_energy_cells,current_player,max_score,min_score=state
    if current_player=='MAX':
        new_A_pos=get_new_position(A_pos,move)
        if not is_valid_move(new_A_pos,B_pos):
            return None
        new_remaining_energy_cells=remaining_energy_cells.copy()
        if new_A_pos in new_remaining_energy_cells:
            new_remaining_energy_cells.remove(new_A_pos)
            max_score+=10
        new_state=(new_A_pos,B_pos,new_remaining_energy_cells,'MIN',max_score,min_score)
        return new_state
    elif current_player=='MIN':
            new_B_pos=get_new_position(B_pos,move)
            if not is_valid_move(new_B_pos,A_pos):
                return None
            new_remaining_energy_cells=remaining_energy_cells.copy()
            if new_B_pos in new_remaining_energy_cells:
                new_remaining_energy_cells.remove(new_B_pos)
                min_score+=10
            new_state=(A_pos,new_B_pos,new_remaining_energy_cells,'MAX',max_score,min_score)
            return new_state

In [21]:
def get_successors(state,stats):
    successors=[]
    for move in moves.keys():
        new_state=successor(state,move)
        if new_state is not None:
            successors.append((move,new_state))
            stats['nodes_generated']+=1
    return successors

In [22]:
def manhattan_distance(pos1,pos2):
    return abs(pos1[0]-pos2[0])+abs(pos1[1]-pos2[1])

def positional_advantage(A_pos,B_pos,remaining_energy_cells):
    if not remaining_energy_cells:
        return 0
    A_distances=[manhattan_distance(A_pos,cell) for cell in remaining_energy_cells]
    B_distances=[manhattan_distance(B_pos,cell) for cell in remaining_energy_cells]
    return min(B_distances)-min(A_distances)

In [23]:
def evaluation_function(state):
    A_pos,B_pos,remaining_energy_cells,current_player,max_score,min_score=state
    return (max_score-min_score)+positional_advantage(A_pos,B_pos,remaining_energy_cells)

In [24]:
def minimax(state,depth,stats):
    A_pos,B_pos,remaining_energy_cells,current_player,max_score,min_score=state
    if depth==0 or not remaining_energy_cells:
        return evaluation_function(state)
    stats['nodes_expanded']+=1
    
    successors=get_successors(state,stats)
    
    if current_player=='MAX':
        best_score=float('-inf')
        for move,new_state in successors:
            score=minimax(new_state,depth-1,stats)
            best_score=max(best_score,score)
        return best_score
    
    else:
        best_score=float('inf')
        for move,new_state in successors:
            score=minimax(new_state,depth-1,stats)
            best_score=min(best_score,score)
        return best_score
    

In [25]:
import time
stats={'nodes_generated':0,'nodes_expanded':0}

start_time=time.perf_counter()

successors=get_successors(initial_state,stats)
current_player=starting_player

if current_player=='MAX':
    best_score=float('-inf')
    best_move=None
    for move,new_state in successors:
        score=minimax(new_state,depth-1,stats)
        if score>best_score:
            best_score=score
            best_move=move
else:
    best_score=float('inf')
    best_move=None
    for move,new_state in successors:
        score=minimax(new_state,depth-1,stats)
        if score<best_score:
            best_score=score
            best_move=move

end_time=time.perf_counter()


print("Best Move: ",best_move)
print("Best Score: ",best_score)
print("Nodes Generated: ",stats['nodes_generated'])
print("Nodes Expanded: ",stats['nodes_expanded'])
print("Execution Time: ",end_time-start_time," seconds")
    

Best Move:  RIGHT
Best Score:  1
Nodes Generated:  66
Nodes Expanded:  30
Execution Time:  0.0011084999423474073  seconds


In [26]:
def alpha_beta(state,depth,alpha,beta,stats ):
    A_pos,B_pos,remaining_energy_cells,current_player,max_score,min_score=state
    if depth==0 or not remaining_energy_cells:
        return evaluation_function(state)
    stats['nodes_expanded']+=1
    
    successors=get_successors(state,stats)
    
    if current_player=='MAX':
        best_score=float('-inf')
        for move,new_state in successors:
            score=alpha_beta(new_state,depth-1,alpha,beta,stats)
            best_score=max(best_score,score)
            alpha=max(alpha,best_score)
            if beta<=alpha:
                stats['nodes_pruned'] += 1
                break
        return best_score
    
    else:
        best_score=float('inf')
        for move,new_state in successors:
            score=alpha_beta(new_state,depth-1,alpha,beta,stats)
            best_score=min(best_score,score)
            beta=min(beta,best_score)
            if beta<=alpha:
                stats['nodes_pruned'] += 1
                break
        return best_score

In [27]:
import time
stats={'nodes_generated':0,'nodes_expanded':0,'nodes_pruned':0}

alpha=float('-inf')
beta=float('inf')

start_time=time.perf_counter()

successors=get_successors(initial_state,stats)
current_player=starting_player

if current_player=='MAX':
    best_score=float('-inf')
    best_move=None
    for move,new_state in successors:
        score=alpha_beta(new_state,depth-1,alpha,beta,stats)
        if score>best_score:
            best_score=score
            best_move=move
            
        alpha=max(alpha,best_score)
else:
    best_score=float('inf')
    best_move=None
    for move,new_state in successors:
        score=alpha_beta(new_state,depth-1,alpha,beta,stats)
        if score<best_score:
            best_score=score
            best_move=move
        beta=min(beta,best_score)

end_time=time.perf_counter()


print("Best Move: ",best_move)
print("Best Score: ",best_score)
print("Nodes Generated: ",stats['nodes_generated'])
print("Nodes Expanded: ",stats['nodes_expanded'])
print("Nodes Pruned: ",stats['nodes_pruned'])
print("Execution Time: ",end_time-start_time," seconds")
    

Best Move:  RIGHT
Best Score:  1
Nodes Generated:  44
Nodes Expanded:  19
Nodes Pruned:  7
Execution Time:  0.0007881000638008118  seconds


In [28]:
def get_heuristic_successors(state,stats):
    successors=get_successors(state,stats)
    energy_moves=[]
    normal_moves=[]
    
    A_pos,B_pos,remaining_energy_cells,current_player,max_score,min_score=state
    
    for move,new_state in successors:
        if current_player=="MAX":
            if new_state[4]>max_score:
                energy_moves.append((move,new_state))
            else:
                normal_moves.append((move,new_state))
        else:
            if new_state[5]>min_score:
                energy_moves.append((move,new_state))
            else:
                normal_moves.append((move,new_state))
    return energy_moves+normal_moves

In [29]:
def alpha_beta_with_heuristic(state,depth,alpha,beta,stats ):
    A_pos,B_pos,remaining_energy_cells,current_player,max_score,min_score=state
    if depth==0 or not remaining_energy_cells:
        return evaluation_function(state)
    stats['nodes_expanded']+=1
    
    successors=get_heuristic_successors(state,stats)
    
    if current_player=='MAX':
        best_score=float('-inf')
        for move,new_state in successors:
            score=alpha_beta_with_heuristic(new_state,depth-1,alpha,beta,stats)
            best_score=max(best_score,score)
            alpha=max(alpha,best_score)
            if beta<=alpha:
                stats['nodes_pruned'] += 1
                break
        return best_score
    
    else:
        best_score=float('inf')
        for move,new_state in successors:
            score=alpha_beta_with_heuristic(new_state,depth-1,alpha,beta,stats)
            best_score=min(best_score,score)
            beta=min(beta,best_score)
            if beta<=alpha:
                stats['nodes_pruned'] += 1
                break
        return best_score

In [30]:
import time
stats={'nodes_generated':0,'nodes_expanded':0,'nodes_pruned':0}

alpha=float('-inf')
beta=float('inf')

start_time=time.perf_counter()

successors=get_heuristic_successors(initial_state,stats)
current_player=starting_player

if current_player=='MAX':
    best_score=float('-inf')
    best_move=None
    for move,new_state in successors:
        score=alpha_beta_with_heuristic(new_state,depth-1,alpha,beta,stats)
        if score>best_score:
            best_score=score
            best_move=move
            
        alpha=max(alpha,best_score)
else:
    best_score=float('inf')
    best_move=None
    for move,new_state in successors:
        score=alpha_beta_with_heuristic(new_state,depth-1,alpha,beta,stats)
        if score<best_score:
            best_score=score
            best_move=move
        beta=min(beta,best_score)

end_time=time.perf_counter()


print("Best Move: ",best_move)
print("Best Score: ",best_score)
print("Nodes Generated: ",stats['nodes_generated'])
print("Nodes Expanded: ",stats['nodes_expanded'])
print("Nodes Pruned: ",stats['nodes_pruned'])
print("Execution Time: ",end_time-start_time," seconds")
    

Best Move:  RIGHT
Best Score:  1
Nodes Generated:  44
Nodes Expanded:  19
Nodes Pruned:  7
Execution Time:  0.0007730999495834112  seconds


## Comparison of Minimax and Alpha-Beta Variants

| Parameter | Minimax | Alpha-Beta (Normal Ordering) | Alpha-Beta (Heuristic Ordering) |
|---|---:|---:|---:|
| **Best Move** | RIGHT | RIGHT | RIGHT |
| **Best Score** | 1 | 1 | 1 |
| **Nodes Generated** | 66 | 44 | 44 |
| **Nodes Expanded** | 30 | 19 | 19 |
| **Nodes Pruned** | N/A | 7 | 7 |
| **Execution Time (s)** | 0.0011085 | 0.0007881 | 0.0007731 |
| **Optimal Solution** | Yes | Yes | Yes |

## Node Reduction Compared with Minimax

| Algorithm | Nodes Generated | Reduction | Nodes Expanded | Reduction |
|---|---:|---:|---:|---:|
| **Minimax** | 66 | — | 30 | — |
| **Alpha-Beta (Normal Ordering)** | 44 | 33.33% | 19 | 36.67% |
| **Alpha-Beta (Heuristic Ordering)** | 44 | 33.33% | 19 | 36.67% |

## Observation

- All three algorithms produce the same **Best Move (`RIGHT`)** and **Best Score (`1`)**.
- Alpha-Beta pruning reduces the number of nodes explored compared with standard Minimax.
- Alpha-Beta with normal ordering and heuristic ordering produced the **same node counts** for this particular game tree.
- Heuristic ordering was marginally faster in this run, with an execution time of **0.0007731 seconds**.
- The results demonstrate that Alpha-Beta pruning can improve search efficiency without changing the optimal decision.